# Linear Regression Model Selection

This notebook loads the sample regression data, chooses the best single-feature model, then builds a 2-feature model and saves both versions under a versioned registry.

In [ ]:
from itertools import combinations
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
root = Path.cwd()
data_path = root / 'data' / 'raw' / 'sampregdata.csv'
df = pd.read_csv(data_path)
df.head()

In [ ]:
x_cols = [c for c in df.columns if c.startswith('x')]
best_single = None
for col in x_cols:
    model = LinearRegression()
    X = df[[col]]
    y = df['y']
    model.fit(X, y)
    score = r2_score(y, model.predict(X))
    record = {'feature': col, 'r2': score, 'coef': float(model.coef_[0]), 'intercept': float(model.intercept_)}
    if best_single is None or score > best_single['r2']:
        best_single = record

best_single

In [ ]:
best_pair = None
for combo in combinations(x_cols, 2):
    model = LinearRegression()
    X = df[list(combo)]
    y = df['y']
    model.fit(X, y)
    score = r2_score(y, model.predict(X))
    record = {'features': list(combo), 'r2': score, 'coef': model.coef_.tolist(), 'intercept': float(model.intercept_)}
    if best_pair is None or score > best_pair['r2']:
        best_pair = record

best_pair

In [ ]:
models_dir = root / 'models'
models_dir.mkdir(exist_ok=True)

single_model = LinearRegression()
single_model.fit(df[[best_single['feature']]], df['y'])
v1_dir = models_dir / 'v1_single_feature'
v1_dir.mkdir(exist_ok=True)
joblib.dump(single_model, v1_dir / 'model.joblib')
(v1_dir / 'metadata.json').write_text(json.dumps({
    'version': 'v1',
    'model_type': 'LinearRegression',
    'features': [best_single['feature']],
    'r2': best_single['r2'],
    'coef': [best_single['coef']],
    'intercept': best_single['intercept']
}, indent=2), encoding='utf-8')

pair_model = LinearRegression()
pair_model.fit(df[list(best_pair['features'])], df['y'])
v2_dir = models_dir / 'v2_two_feature'
v2_dir.mkdir(exist_ok=True)
joblib.dump(pair_model, v2_dir / 'model.joblib')
(v2_dir / 'metadata.json').write_text(json.dumps({
    'version': 'v2',
    'model_type': 'LinearRegression',
    'features': best_pair['features'],
    'r2': best_pair['r2'],
    'coef': best_pair['coef'],
    'intercept': best_pair['intercept'],
    'current': True
}, indent=2), encoding='utf-8')

(models_dir / 'current_model.json').write_text(json.dumps({
    'current_version': 'v2',
    'current_model_path': 'models/v2_two_feature',
    'previous_version': 'v1',
    'previous_model_path': 'models/v1_single_feature'
}, indent=2), encoding='utf-8')